### Connexion au blob storage

In [0]:
# Connexion au blob storage
spark.conf.set("fs.azure.account.key.blobstoragehandson.blob.core.windows.net", "x")

### Import des parquets dans un dataframe

###### TV

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType, TimestampType
from pyspark.sql import functions as F
from datetime import datetime
from dateutil.relativedelta import relativedelta

# Paramètres
container = "next-best-offer"
storage_account = "blobstoragehandson"
sub_folder = "raw_data/tv/*/tv.parquet"

endpoint = f"wasbs://{container}@{storage_account}.blob.core.windows.net/{sub_folder}"

# Définition du schéma
tv_schema = StructType([
    StructField("CUST_NUM", LongType(), True),
    StructField("EVT_DT", TimestampType(), True),
    StructField("TV_PROGRAM", StringType(), True),
    StructField("DURATION", LongType(), True)
])

# Lire tous les fichiers parquet
df_tv = spark.read.schema(tv_schema).parquet(endpoint)

# Identifier la date max réelle
max_date = df_tv.agg(F.max("EVT_DT")).collect()[0][0]
print("Dernière date disponible :", max_date)

# Déterminer le mois M-1 (dernier mois complet)
last_month_date = datetime(max_date.year, max_date.month, 1) - relativedelta(months=1)
year_month_target = (last_month_date.year, last_month_date.month)

print("Mois sélectionné (M-1) :", year_month_target)

# Ajouter colonnes année et mois TEMPORAIRES pour le filtrage
df_tv = df_tv.withColumn("_year_tmp", F.year("EVT_DT")) \
             .withColumn("_month_tmp", F.month("EVT_DT"))

# Filtrer pour garder uniquement M-1
df_tv = df_tv.filter(
    (F.col("_year_tmp") == year_month_target[0]) &
    (F.col("_month_tmp") == year_month_target[1])
)

# Supprimer les colonnes temporaires
df_tv = df_tv.drop("_year_tmp", "_month_tmp")

In [0]:
df_tv.dropDuplicates()

### Feature Engineering

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import to_date, month, year

def preprocess_df(df: DataFrame, duration_col: str = None) -> DataFrame:
    """
    Standardise un DataFrame pour ton pipeline ML :
    - Convertit la colonne EVT_DT en Date
    - Crée les colonnes MONTH et YEAR
    - Renomme la colonne DURATION si spécifiée
    - Supprime systématiquement EVT_DT après création des colonnes MONTH/YEAR

    Args:
        df (DataFrame): DataFrame Spark
        duration_col (str, optional): nom pour renommer la colonne DURATION

    Returns:
        DataFrame: DataFrame traité
    """
    # Convertir EVT_DT en date
    df = df.withColumn("EVT_DT", to_date("EVT_DT"))
    
    # Renommer la colonne DURATION si nécessaire
    if duration_col:
        df = df.withColumnRenamed("DURATION", duration_col)
    
    # Créer MONTH et YEAR
    df = df.withColumn("MONTH", month("EVT_DT")) \
           .withColumn("YEAR", year("EVT_DT")) #\
           #.drop("EVT_DT")
    
    return df

In [0]:
df_tv       = preprocess_df(df_tv, duration_col="DURATION_TV")

##### TV

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.functions import col, lit, struct, max as F_max, countDistinct, percent_rank
from dateutil.relativedelta import relativedelta
from pyspark.sql import DataFrame

def process_l3m_tv(df_tv: DataFrame, n_months: int = 3) -> DataFrame:
    """
    Calcul L3M pour df_tv : durée, jours distincts, rank, pivot et renommage.
    
    Args:
        df_tv (DataFrame): DataFrame TV avec EVT_DT et DURATION_TV
        n_months (int): nombre de mois pour L3M
    
    Returns:
        DataFrame: pivoté, rang percentile calculé et colonnes renommées
    """
    
    # Date max et période L3M
    date_max = df_tv.agg(F.max("EVT_DT")).first()[0]
    month_year_l3m = [
        ((date_max - relativedelta(months=i)).month, (date_max - relativedelta(months=i)).year)
        for i in range(n_months)
    ]
    df_tv = df_tv.filter(
        struct("MONTH", "YEAR").isin([struct(lit(m), lit(y)) for m, y in month_year_l3m])
    )

    # Agrégation durée + jours distincts
    df_agg = df_tv.groupBy("CUST_NUM", "TV_PROGRAM", "MONTH", "YEAR").agg(
        F.sum("DURATION_TV").alias("TV_DURATION_L3M"),
        F.countDistinct("EVT_DT").alias("TV_NB_DAYS_USAGE_L3M")
    )

    # Rang percentile par programme
    window_spec = Window.partitionBy("TV_PROGRAM", "MONTH", "YEAR").orderBy("TV_DURATION_L3M")
    df_agg = df_agg.withColumn("PERCENT_RANK", percent_rank().over(window_spec)) \
                   .withColumn("TV_RANK", (col("PERCENT_RANK") * 99 + 1).cast("int")) \
                   .drop("PERCENT_RANK")

    # Pivot
    df_pivoted = df_agg.groupBy("CUST_NUM", "MONTH", "YEAR").pivot("TV_PROGRAM").agg(
        F_max("TV_DURATION_L3M"),
        F_max("TV_NB_DAYS_USAGE_L3M"),
        F_max("TV_RANK")
    )

    # Renommage dynamique des colonnes
    renamed_columns = []
    for c in df_pivoted.columns:
        if c in ["CUST_NUM", "MONTH", "YEAR"]:
            continue
        if "_max(TV_DURATION_L3M)" in c:
            cat = c.replace("_max(TV_DURATION_L3M)", "")
            new_col = f"TV_DURATION_{cat}_L3M_SECONDES"
        elif "_max(TV_NB_DAYS_USAGE_L3M)" in c:
            cat = c.replace("_max(TV_NB_DAYS_USAGE_L3M)", "")
            new_col = f"TV_DURATION_{cat}_L3M_DAYS"
        elif "_max(TV_RANK)" in c:
            cat = c.replace("_max(TV_RANK)", "")
            new_col = f"TV_RANK_{cat}_L3M"
        else:
            new_col = c
        renamed_columns.append((c, new_col))

    for old_col, new_col in renamed_columns:
        df_pivoted = df_pivoted.withColumnRenamed(old_col, new_col)

    # Supprimer EVT_DT car plus utile après calcul
    df_result = df_pivoted.drop("EVT_DT")

    return df_result

In [0]:
df_tv = process_l3m_tv(df_tv)

### Conversion du dataframe en delta table + test

In [0]:
%sql
DROP TABLE IF EXISTS nboTVAggregateM_1;

In [0]:
# Sauver le dataframe au format delta table natif de Databricks
df_tv.write.format("delta").saveAsTable("nboTVAggregateM_1")

In [0]:
%sql
-- Test ouverture en SQL de la delta table
SELECT * 
FROM nboTVAggregateM_1;